# 03 · 멀티 모델 벤치마크

YAML config 기반으로 여러 모델을 동일 조건에서 훈련·평가하고 리더보드를 생성합니다.

**추천 config**:
- 빠른 검증: `colab_quick.yaml` (5 epochs, 25% 데이터)
- Colab 전체: `colab_full.yaml` (20 epochs, 100% 데이터)
- 논문 기준: `benchmark_baseline.yaml` (45 epochs, 100% 데이터)

## ① 파라미터

In [ ]:
from pathlib import Path

# ── 수정 가능한 파라미터 ────────────────────────────────────────────
REPO_DIR       = Path(globals().get('REPO_DIR',       '/content/Military_Object_Detection'))
WORKSPACE_ROOT = Path(globals().get('WORKSPACE_ROOT', '/content/drive/MyDrive/Military_Object_Detection'))
CACHE_ROOT     = Path(globals().get('CACHE_ROOT',     globals().get('MAD_DATA_CACHE_ROOT', '/content/.cache/mad')))
KAGGLE_DATASET_ID = globals().get('KAGGLE_DATASET_ID', 'a2015003713/militaryaircraftdetectiondataset')

# 어떤 config를 사용할지 선택 (아래 중 하나 주석 해제)
CONFIG_PATH = Path(globals().get('CONFIG_PATH',
                   str(REPO_DIR / 'configs' / 'colab_full.yaml')))
#CONFIG_PATH = REPO_DIR / 'configs' / 'colab_quick.yaml'       # 빠른 검증
#CONFIG_PATH = REPO_DIR / 'configs' / 'benchmark_baseline.yaml' # 논문 실험

DATASET_YAML   = Path(globals().get('DATASET_YAML',
                       str(CACHE_ROOT / 'datasets' / 'a2015003713__militaryaircraftdetectiondataset' / 'yolo_dataset' / 'dataset.yaml')))
OUTPUT_DIR     = Path(globals().get('OUTPUT_DIR',
                       str(WORKSPACE_ROOT / 'experiments' / Path(CONFIG_PATH).stem)))

# 선택적 오버라이드 (None이면 config 값 사용)
OVERRIDE_EPOCHS   = globals().get('OVERRIDE_EPOCHS',   None)
OVERRIDE_IMGSZ    = globals().get('OVERRIDE_IMGSZ',    None)
OVERRIDE_BATCH    = globals().get('OVERRIDE_BATCH',    None)
OVERRIDE_FRACTION = globals().get('OVERRIDE_FRACTION', None)
WANDB_MODE        = globals().get('WANDB_MODE', 'off')  # 'off' | 'on'
# ──────────────────────────────────────────────────────────────────


## ② 환경 초기화 및 GPU 확인

In [ ]:
import sys, os
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
os.chdir(REPO_DIR)
os.environ['MAD_WORKSPACE_ROOT'] = str(WORKSPACE_ROOT)
os.environ['MAD_DATA_CACHE_ROOT'] = str(CACHE_ROOT)

from mad.colab_utils import setup_colab_env, check_gpu, check_dataset

setup_colab_env(REPO_DIR, WORKSPACE_ROOT)
check_gpu(require=False)
check_dataset(DATASET_YAML)


## ③ 벤치마크 실행

> ⚠️ 훈련 시간이 길 수 있습니다. Colab 탭을 닫아도 실행은 유지됩니다.

In [ ]:
from mad.benchmark import run_benchmark

overrides = {
    'workspace_root': WORKSPACE_ROOT,
    'dataset_yaml':   DATASET_YAML,
    'output_dir':     OUTPUT_DIR,
    'train.epochs':   OVERRIDE_EPOCHS,
    'train.imgsz':    OVERRIDE_IMGSZ,
    'train.batch':    OVERRIDE_BATCH,
    'train.fraction': OVERRIDE_FRACTION,
    'wandb.enabled': True if WANDB_MODE == 'on' else False,
}

summary = run_benchmark(CONFIG_PATH, overrides=overrides)

print('\n✅ 벤치마크 완료')
print(f"   study_dir       : {summary.get('study_dir')}")
print(f"   total_runs      : {summary.get('total_runs')}")
print(f"   successful_runs : {summary.get('successful_runs')}")
print(f"   failed_runs     : {summary.get('failed_runs')}")

## ④ 리더보드 확인

In [ ]:
from pathlib import Path
import pandas as pd

study_dir = Path(summary['study_dir'])
summaries = study_dir / 'summaries'

# 전체 run 결과
all_runs_csv = summaries / 'benchmark_all_runs.csv'
if all_runs_csv.exists():
    df = pd.read_csv(all_runs_csv)
    cols = [c for c in ['model_id', 'seed', 'checkpoint_type',
                         'val_map50_95', 'test_map50_95',
                         'val_map50', 'test_map50', 'status'] if c in df.columns]
    display(df[cols])

# 모델별 요약 (mean ± std)
model_summary_csv = summaries / 'benchmark_model_summary.csv'
if model_summary_csv.exists():
    print('\n--- 모델별 집계 요약 ---')
    display(pd.read_csv(model_summary_csv))

# 리더보드 마크다운
leaderboard_md = summaries / 'leaderboard.md'
if leaderboard_md.exists():
    print('\n--- Leaderboard ---')
    print(leaderboard_md.read_text(encoding='utf-8'))

## CLI 동등 명령어
```bash
# Colab 터미널 또는 !셀에서 실행
!python scripts/run_benchmark.py \\
  --config configs/colab_full.yaml \\
  --dataset-yaml $DATASET_YAML \\
  --output-dir $MAD_WORKSPACE_ROOT/experiments/colab_full \\
  --wandb off
```